CONNECT YOUR PROJECT TO YOUR SQL SERVER AND IMPORT YOUR LIBRARIES

In [ ]:
#i didn't install/uv add seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import requests
import psycopg2
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from sklearn.linear_model import LinearRegression
from sqlalchemy import create_engine



In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")


In [ ]:
conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)

LOAD DATA FROM POSTGRESQL

In [ ]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [ ]:
food_prices = pd.read_sql("SELECT * FROM food_prices", engine)

In [ ]:
fx_rates = pd.read_sql("SELECT * FROM fx_rates", engine)

In [ ]:
food_prices.head()

In [ ]:
fx_rates.head()

First, aggregate fx data to be on a monthly basis so we can analyse fairly

We can do aggregate the daily fx rates using
1. Monthly average: Typical exchange rate during that month. 
    Good if hypothesis is: food prices respond to general exchnage rate conditions over then month
2. End-of-month close: Last fx rate of each month. 
    Good if food report reflects month-end pricing
3. Monthly median: More robust if there are wierd spikes/outliers

We'll be using the monthly average for this project.

Daily foreign exchange observations were aggregated into monthly frequency of food price data published by the National Bureau of Statistics.

We're turning the 'date' column to datetime format so pandas read it correctly. 
Then setting date as the index, which tells pandas to use the date as the timeline for this data.
Resample then sets the month (M) to group all rows by month

In [ ]:
print(type(fx_rates))

In [ ]:
print(type(fx_rates['usd_ngn'].iloc[0]))
print(fx_rates['usd_ngn'].iloc[0])

In [ ]:
fx_rates.info()

In [ ]:
if 'date' not in fx_rates.columns:
    fx_rates = fx_rates.reset_index()


In [ ]:
fx_rates['date'] = pd.to_datetime(fx_rates['date'])

In [ ]:
#fx_rates = fx_rates.set_index('date') #this tells pandas to use the date as the timeline for this data. 
#becuase resample only works when time is the index
monthly_fx = fx_rates.resample('ME', on='date')['usd_ngn'].mean().reset_index() #resammple groups all rows by month. ('M')
 #without reset index, date is stuck as index and merging with food is harder

In [ ]:
monthly_fx['date'] = pd.to_datetime(monthly_fx['date'])

In [ ]:
monthly_fx.head()

In [ ]:
monthly_fx.shape

In [ ]:
monthly_fx.tail()

In [ ]:
monthly_fx = monthly_fx.iloc[1:-4]

In [ ]:
monthly_fx.tail()

In [ ]:
monthly_fx.head()

In [ ]:
monthly_fx.shape

In [ ]:
#create matching month key in forex table
monthly_fx['date'] = pd.to_datetime(monthly_fx['date'])
monthly_fx['month'] = monthly_fx['date'].dt.to_period('M')

#create matching month key in food table
food_prices['date'] = pd.to_datetime(food_prices['date'])
food_prices['month'] = food_prices['date'].dt.to_period('M')

#merge
merged_df = food_prices.merge(
    monthly_fx[['month', 'usd_ngn']],
    on='month',
    how='left'
)

In [ ]:
merged_df.head()

In [ ]:
merged_df.tail()

ANALYSIS TIME

We are not gonna be doing train/test split. That only works when the goal is to predict food price given exchange rate and vice versa. Right now we're doing exploratory data analysis. 

In [ ]:
merged_df[['avg_price', 'usd_ngn']].corr()

In [ ]:
y = merged_df['avg_price']
x = merged_df ['usd_ngn']
plt.scatter(x, y)
plt.title("Scatterplot of Exchange Rate vs. Avg. Food Prices")
plt.xlabel("usd_ngn")
plt.ylabel("avg_price")

By using all food prices together for the .corr and scatterplot function, I've averaged all the flunctuations away/ Check each by food

In [ ]:
food_corrs = []
for food in merged_df['item'].unique():
    subset = merged_df[merged_df['item'] == food]
    corr = subset['avg_price'].corr(subset['usd_ngn'])

    food_corrs.append({
    'item': food,
    'correlation': corr
    })
corr_df = pd.DataFrame(food_corrs)
corr_df = corr_df.dropna()
print(corr_df.sort_values('correlation', ascending=False))

Creating a bar chart of the food items with the highest correlation to exchange rate flunctuation

In [ ]:
top_corr = corr_df.sort_values(
    by='correlation',
    ascending=False
).head(10)

plt.figure(figsize=(12, 6))

plt.bar(top_corr['item'], top_corr['correlation'])

plt.title('Top 10 Food Items Most Correlated with USD/NGN')

plt.xlabel('Food Item')

plt.ylabel('Correlation')

plt.xticks(rotation=45, ha='right')

plt.tight_layout()

plt.show()

In [ ]:
merged_df['item'].unique()

Creating a scatterplot of the food item with the single highest fx correlation

In [ ]:
import numpy as np

wheat_df = merged_df[
    merged_df['item'] == 'Wheat Flour, Prepacked (2kg)'
]

x = wheat_df['usd_ngn']
y = wheat_df['avg_price']

plt.figure(figsize=(8, 5))

plt.scatter(x, y)

m, b = np.polyfit(x, y, 1)

plt.plot(x, m*x + b)

plt.title("Exchange Rate vs Wheat Flour Prices")

plt.xlabel("USD/NGN Exchange Rate")

plt.ylabel("Wheat Flour Average Price")

plt.tight_layout()

plt.show()

Creating a scatterplot which shows the contrast between exchange rate correlation of two different items

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Wheat flour
wheat_df = merged_df[
    merged_df['item'] == 'Wheat Flour, Prepacked (2kg)'
]

# Chicken feet
chicken_df = merged_df[
    merged_df['item'] == 'Chicken feet'
]

# Variables
x1 = wheat_df['usd_ngn']
y1 = wheat_df['avg_price']

x2 = chicken_df['usd_ngn']
y2 = chicken_df['avg_price']

# Plot
plt.figure(figsize=(10, 6))

# Scatterplots
plt.scatter(x1, y1, label='Wheat Flour')
plt.scatter(x2, y2, label='Chicken feet')

# Trendline for wheat
m1, b1 = np.polyfit(x1, y1, 1)
plt.plot(x1, m1*x1 + b1)

# Trendline for chicken
m2, b2 = np.polyfit(x2, y2, 1)
plt.plot(x2, m2*x2 + b2)

# Labels
plt.title("USD/NGN vs Food Prices")

plt.xlabel("USD/NGN Exchange Rate")

plt.ylabel("Average Food Price")

plt.legend()

plt.tight_layout()

plt.show()

IMPORT PMS DATA TO CHECK FOR CORRELATION BETWEEN PMS PRICES AND FOOD PRICES

In [ ]:
pms_data = pd.read_csv('pms_data.csv')

In [ ]:
pms_data.head()

In [ ]:
print(merged_df["month"].dtype)
print(pms_data["month"].dtype)

Setting index to 'month' and merging the pms dataframe with the former merged dataframe

In [ ]:
pms_data["month"] = pd.PeriodIndex(
    pms_data["month"],
    freq="M"
)

In [ ]:
print(merged_df["month"].dtype)
print(pms_data["month"].dtype)

In [ ]:
merged_df = pd.merge(
    merged_df,
    pms_data,
    on="month",
    how="left"
)
merged_df.head()

In [ ]:
merged_df.tail()

In [ ]:
# Calculate food-price vs PMS-price correlations

food_corrs_pms = []

for food in merged_df['item'].unique():

    subset = merged_df[
        merged_df['item'] == food
    ]

    corr = subset['avg_price'].corr(
        subset['avg_pms_price']
    )

    food_corrs_pms.append({
        'item': food,
        'pms_corr': corr
    })

corr_pms_df = pd.DataFrame(food_corrs_pms)

corr_pms_df = corr_pms_df.dropna()

print(
    corr_pms_df.sort_values(
        'pms_corr',
        ascending=False
    )
)

In [ ]:
# Assuming corr_df already exists from your FX analysis
# and contains columns:
# item, correlation

corr_df = corr_df.rename(
    columns={'correlation': 'fx_corr'}
)

# Merge FX and PMS correlations

comparison_df = pd.merge(
    corr_df,
    corr_pms_df,
    on='item'
)

# Compare PMS influence against FX influence

comparison_df['pms_minus_fx'] = (
    comparison_df['pms_corr']
    - comparison_df['fx_corr']
)

# Foods whose prices track PMS more than FX

comparison_df = comparison_df.sort_values(
    'pms_minus_fx',
    ascending=False
)

print(comparison_df)

Plotting egg prices again pms values

In [ ]:
eggs_df = merged_df[
    merged_df['item'] == 'Agric hen eggs, (a Crate of 30 pieces)'
]

plt.figure(figsize=(8,5))

plt.scatter(
    eggs_df['avg_pms_price'],
    eggs_df['avg_price']
)

m, b = np.polyfit(
    eggs_df['avg_pms_price'],
    eggs_df['avg_price'],
    1
)

plt.plot(
    eggs_df['avg_pms_price'],
    m*eggs_df['avg_pms_price'] + b
)

plt.xlabel("Average PMS Price")
plt.ylabel("Egg Price")

plt.title("Egg Prices vs PMS Prices")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    eggs_df['usd_ngn'],
    eggs_df['avg_price']
)

m, b = np.polyfit(
    eggs_df['usd_ngn'],
    eggs_df['avg_price'],
    1
)

plt.plot(
    eggs_df['usd_ngn'],
    m*eggs_df['usd_ngn'] + b
)

plt.xlabel("USD/NGN")
plt.ylabel("Egg Price")

plt.title("Egg Prices vs FX")

plt.show()

In [ ]:
merged_df.groupby('item').size()

In [ ]:
merged_df.to_excel('food_fx_analysis.xlsx', index=False)